# nb02 - SQL and pandas parity: CMS Star Ratings

Every figure on the Tableau dashboard **L.A. Care Medicare Scorecard** is reproduced here twice, once in DuckDB SQL and once in pandas, and asserted against the number Tableau actually renders. If an assert fails, the blog post is wrong.

Published dashboard: https://public.tableau.com/views/medicare_star_ratings/L_A_CareMedicareScorecard

**The point of this notebook is not the numbers, it is the translation.** Each Tableau feature has an exact SQL analog:

| Tableau | SQL | pandas |
|---|---|---|
| `{ FIXED [Year] : AVG([Overall Stars]) }` | `AVG(...) OVER (PARTITION BY Year)` | `groupby('Year')[...].transform('mean')` |
| `{ FIXED [Year], [Measure Code] : AVG([Stars]) }` | `AVG(...) OVER (PARTITION BY Year, MeasureCode)` | `groupby(['Year','MeasureCode'])[...].transform('mean')` |
| Add to Context | move the condition into the **CTE**, so it runs before the window | filter the frame **before** the groupby |
| Difference From, Relative to First | `value - FIRST_VALUE(value) OVER (PARTITION BY org ORDER BY Year)` | `g.transform(lambda v: v - v.iloc[0])` |
| Group (`Parent Org (group)`) | `CASE WHEN parent = 'x' THEN 'y' ELSE parent END` | `.replace({...})` |
| Alias | no SQL analog, it is display only | no pandas analog |

**Data note.** `nb01` drops the CMS measurement-period row (137 rows with no contract id, which `star_to_num` turned into fake 0.0-star contracts) and adds `ParentOrgGroup`, which merges the duplicate CMS spellings of Molina and Samaritan. Both changes are required for parity: without the first, every national benchmark is low by up to 0.013 stars; without the second, Molina splits into two companies and drops out of the parent-org chart.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DATA = Path('..') / 'data'
summary  = pd.read_csv(DATA / 'star_summary_clean.csv')
measures = pd.read_csv(DATA / 'star_measures_clean.csv')

con = duckdb.connect()
con.register('summary', summary)
con.register('measures', measures)

LA   = 'H1224'   # L.A. Care Health Plan
YEAR = 2026

print('summary ', summary.shape)
print('measures', measures.shape)

RESULTS = []
def check(name, tableau, sql, pd_, places=3):
    """Assert the SQL and pandas answers both match what Tableau renders."""
    ok = (round(float(sql), places) == round(float(tableau), places)
          and round(float(pd_), places) == round(float(tableau), places))
    RESULTS.append((name, tableau, sql, pd_, ok))
    flag = 'PASS' if ok else 'FAIL'
    print(f'[{flag}] {name}: tableau={tableau} sql={round(float(sql), places)} pandas={round(float(pd_), places)}')
    assert ok, name

summary  (2415, 11)
measures (110321, 11)


## 1. `National Avg Star` - the FIXED LOD benchmark

Tableau: `National Avg Star = { FIXED [Year] : AVG([Overall Stars]) }`, used on the `Stars vs National` sheet as the orange line.

The whole reason this is a FIXED LOD and not a plain average is that the sheet is **filtered to L.A. Care**. A plain `AVG([Overall Stars])` would collapse to L.A. Care's own 3.0. FIXED is computed before the dimension filter runs, so it still sees all 516 rated contracts.

- **SQL approach:** a window function, `AVG(...) OVER (PARTITION BY Year)`, computed in a CTE over the *unfiltered* table, then the contract filter is applied outside it. The CTE boundary is what makes the window blind to the filter, exactly like FIXED.
- **pandas approach:** `groupby('Year').transform('mean')` broadcasts the year-level average back onto every row before any row filtering happens.

Tableau renders 3.7 / 3.6 / 3.7 because the axis is rounded to one decimal. The underlying values are 3.6817 / 3.6497 / 3.6521.

In [2]:
sql = con.execute("""
    WITH benchmark AS (                       -- runs BEFORE the contract filter, like FIXED
        SELECT Year,
               Contract,
               Overall_Stars,
               AVG(Overall_Stars) OVER (PARTITION BY Year) AS national_avg_star
        FROM summary
    )
    SELECT Year, national_avg_star
    FROM benchmark
    WHERE Contract = ?                        -- the sheet filter, applied AFTER the window
    ORDER BY Year
""", [LA]).df()

s = summary.copy()
s['national_avg_star'] = s.groupby('Year')['Overall_Stars'].transform('mean')
pdf = s.loc[s.Contract == LA, ['Year', 'national_avg_star']].sort_values('Year')

print(sql.to_string(index=False))

for yr, tab in [(2024, 3.6817), (2025, 3.6497), (2026, 3.6521)]:
    check(f'National Avg Star {yr}', tab,
          sql.loc[sql.Year == yr, 'national_avg_star'].iloc[0],
          pdf.loc[pdf.Year == yr, 'national_avg_star'].iloc[0], places=4)

 Year  national_avg_star
 2024           3.681651
 2025           3.649712
 2026           3.652132
[PASS] National Avg Star 2024: tableau=3.6817 sql=3.6817 pandas=3.6817
[PASS] National Avg Star 2025: tableau=3.6497 sql=3.6497 pandas=3.6497
[PASS] National Avg Star 2026: tableau=3.6521 sql=3.6521 pandas=3.6521


## 2. `Stars vs National` - L.A. Care's own line

The blue line. L.A. Care is unrated in 2024 (the plan was too new), then flat at 3.0 in 2025 and 2026. The headline `L.A. Care Trails the Nation by 0.7 Stars` is 3.65 minus 3.0, rounded.

- **SQL approach:** a plain filtered select, no window needed. This is the *un*-benchmarked number.
- **pandas approach:** a boolean mask on the same two columns.

In [3]:
sql = con.execute("""
    SELECT Year, Overall_Stars, PartC_Stars, PartD_Stars
    FROM summary
    WHERE Contract = ?
    ORDER BY Year
""", [LA]).df()

pdf = summary.loc[summary.Contract == LA,
                  ['Year', 'Overall_Stars', 'PartC_Stars', 'PartD_Stars']].sort_values('Year')

print(sql.to_string(index=False))

for yr in (2025, 2026):
    check(f'L.A. Care overall star {yr}', 3.0,
          sql.loc[sql.Year == yr, 'Overall_Stars'].iloc[0],
          pdf.loc[pdf.Year == yr, 'Overall_Stars'].iloc[0])

assert pd.isna(sql.loc[sql.Year == 2024, 'Overall_Stars'].iloc[0]), '2024 should be unrated'
print('[PASS] L.A. Care 2024 is unrated (null), as Tableau shows')

gap = 3.6521 - 3.0
print(f'\nheadline check: gap = {gap:.4f} -> rounds to 0.7 stars')
assert round(gap, 1) == 0.7

 Year  Overall_Stars  PartC_Stars  PartD_Stars
 2024            NaN          NaN          NaN
 2025            3.0          2.5          3.5
 2026            3.0          3.0          4.0
[PASS] L.A. Care overall star 2025: tableau=3.0 sql=3.0 pandas=3.0
[PASS] L.A. Care overall star 2026: tableau=3.0 sql=3.0 pandas=3.0
[PASS] L.A. Care 2024 is unrated (null), as Tableau shows

headline check: gap = 0.6521 -> rounds to 0.7 stars


## 3. `Star Distribution 2026` - the LOD scalar and the boolean bridge

Tableau uses two calcs here:

```
LA Care Star    = { FIXED [Year] : MAX(IF [Contract] = "H1224" THEN [Overall Stars] END) }
Beats L.A. Care = [Overall Stars] > [LA Care Star]
```

`LA Care Star` is a FIXED LOD that pulls a **single scalar** (L.A. Care's own star) down onto every row, so each contract can be compared against it. That is the trick worth remembering: a FIXED LOD with a `MAX(IF ...)` inside is how you get "the value for one specific member" onto all rows.

- **SQL approach:** a window `MAX(CASE WHEN contract = 'H1224' THEN stars END) OVER (PARTITION BY Year)` broadcasts L.A. Care's star to every row of that year. Then a simple comparison and a `COUNT(DISTINCT)` per star level.
- **pandas approach:** `groupby('Year').transform('max')` on a masked column does the same broadcast, then `value_counts()` on the star column.

Headline: `L.A. Care Sits in the Bottom Quarter of Rated Plans`, because 382 of 516 rated contracts (74%) score above 3.0.

In [4]:
sql = con.execute("""
    WITH bridged AS (
        SELECT Year, Contract, Overall_Stars,
               MAX(CASE WHEN Contract = ? THEN Overall_Stars END)
                   OVER (PARTITION BY Year) AS la_care_star
        FROM summary
        WHERE Overall_Stars IS NOT NULL          -- unrated contracts are excluded from the chart
    )
    SELECT Overall_Stars,
           COUNT(DISTINCT Contract)                        AS contracts,
           BOOL_OR(Overall_Stars > la_care_star)           AS beats_la_care
    FROM bridged
    WHERE Year = ?
    GROUP BY Overall_Stars
    ORDER BY Overall_Stars
""", [LA, YEAR]).df()

rated = summary[summary.Overall_Stars.notna()].copy()
rated['la_care_star'] = (rated.Overall_Stars.where(rated.Contract == LA)
                              .groupby(rated.Year).transform('max'))
r26 = rated[rated.Year == YEAR]
pdf = (r26.groupby('Overall_Stars')
          .agg(contracts=('Contract', 'nunique'),
               beats_la_care=('Overall_Stars', lambda v: bool((v > 3.0).any())))
          .reset_index())

print(sql.to_string(index=False))

expected = {2.0: 2, 2.5: 21, 3.0: 111, 3.5: 175, 4.0: 116, 4.5: 73, 5.0: 18}
for star, n in expected.items():
    check(f'contracts at {star} stars', n,
          sql.loc[sql.Overall_Stars == star, 'contracts'].iloc[0],
          pdf.loc[pdf.Overall_Stars == star, 'contracts'].iloc[0])

check('total rated contracts 2026', 516, sql.contracts.sum(), pdf.contracts.sum())
check('contracts beating L.A. Care', 382,
      sql.loc[sql.beats_la_care, 'contracts'].sum(),
      pdf.loc[pdf.beats_la_care, 'contracts'].sum())
print(f"\nshare beating L.A. Care: {382/516:.1%}")

 Overall_Stars  contracts  beats_la_care
           2.0          2          False
           2.5         21          False
           3.0        111          False
           3.5        175           True
           4.0        116           True
           4.5         73           True
           5.0         18           True
[PASS] contracts at 2.0 stars: tableau=2 sql=2.0 pandas=2.0
[PASS] contracts at 2.5 stars: tableau=21 sql=21.0 pandas=21.0
[PASS] contracts at 3.0 stars: tableau=111 sql=111.0 pandas=111.0
[PASS] contracts at 3.5 stars: tableau=175 sql=175.0 pandas=175.0
[PASS] contracts at 4.0 stars: tableau=116 sql=116.0 pandas=116.0
[PASS] contracts at 4.5 stars: tableau=73 sql=73.0 pandas=73.0
[PASS] contracts at 5.0 stars: tableau=18 sql=18.0 pandas=18.0
[PASS] total rated contracts 2026: tableau=516 sql=516.0 pandas=516.0
[PASS] contracts beating L.A. Care: tableau=382 sql=382.0 pandas=382.0

share beating L.A. Care: 74.0%


## 4. `Where L.A. Care Loses Stars` - a FIXED LOD at two levels of detail

```
National Avg by Measure = { FIXED [Year], [Measure Code] : AVG([Stars]) }
Gap vs National         = AVG([Stars]) - AVG([National Avg by Measure])
```

Same pattern as check 1, but the LOD now pins **two** dimensions. Every contract row carries the national average for *its* measure in *its* year, so subtracting gives the gap. The sheet is filtered to L.A. Care, and again the FIXED benchmark survives that filter.

**The subtlety that makes this chart 41 bars and not 43.** The LOD is computed at `[Measure Code]`, but the view puts `[Measure Name]` on Rows. Two names are reused across a Part C and a Part D code:

- `Call Center - Foreign Language Interpreter and TTY Availability` (C-side scores 4.0, D-side scores 5.0)
- `Members Choosing to Leave the Plan` (both score 5.0)

So the benchmark is computed per **code**, then the view averages the two codes together per **name**. That is precisely what `AVG([Stars]) - AVG([National Avg by Measure])` says: both terms are aggregated at the view's level of detail. Tableau renders the Call Center bar at +0.213, which is the mean of -0.164 and +0.590. Grouping the SQL by code instead of name gives 43 rows and the wrong bar values.

**Rule:** a FIXED LOD fixes the level the *benchmark* is computed at. It does not fix the level the *view* aggregates at. Reproduce both, in that order.

This is also where the nb01 bug lived: the phantom 0.0-star rows sat inside the `PARTITION BY Year, MeasureCode`, so they pulled every benchmark down. The corrected top bar is **-2.831**, not -2.824.

- **SQL approach:** a CTE computes `AVG(Stars) OVER (PARTITION BY Year, MeasureCode)` over the unfiltered table (the LOD). The outer query filters to L.A. Care and groups by `MeasureName`, averaging both the stars and the benchmark (the view).
- **pandas approach:** `groupby(['Year','MeasureCode']).transform('mean')` for the LOD, then `groupby('MeasureName').agg(mean, mean)` for the view, then subtract.

In [5]:
sql = con.execute("""
    WITH benchmark AS (                                    -- the LOD: level = Year, Measure CODE
        SELECT Year, Contract, MeasureCode, MeasureName, Stars,
               AVG(Stars) OVER (PARTITION BY Year, MeasureCode) AS national_avg_by_measure
        FROM measures
        WHERE Stars IS NOT NULL
    )
    SELECT MeasureName,                                    -- the VIEW: level = Measure NAME
           AVG(Stars)                                       AS la_care_stars,
           AVG(national_avg_by_measure)                     AS national_avg,
           AVG(Stars) - AVG(national_avg_by_measure)        AS gap_vs_national
    FROM benchmark
    WHERE Contract = ? AND Year = ?
    GROUP BY MeasureName
    ORDER BY gap_vs_national
""", [LA, YEAR]).df()

m = measures[measures.Stars.notna()].copy()
m['national_avg_by_measure'] = m.groupby(['Year', 'MeasureCode'])['Stars'].transform('mean')   # LOD
pdf = (m[(m.Contract == LA) & (m.Year == YEAR)]
       .groupby('MeasureName')                                                                  # view
       .agg(la_care_stars=('Stars', 'mean'),
            national_avg=('national_avg_by_measure', 'mean'))
       .reset_index())
pdf['gap_vs_national'] = pdf.la_care_stars - pdf.national_avg
pdf = pdf.sort_values('gap_vs_national')

print(f'{len(sql)} bars (Tableau status bar reads "41 marks")\n')
print(sql.head(8).round(3).to_string(index=False))
print('\n...\n')
print(sql.tail(4).round(3).to_string(index=False))

# the exact bar labels rendered on the Tableau sheet
tableau_bars = {
    'Medication Reconciliation Post-Discharge':                       -2.831,
    'Care Coordination':                                              -2.495,
    'Customer Service':                                               -2.467,
    'Call Center – Foreign Language Interpreter and TTY Availability': 0.213,   # two codes, one bar
    'Members Choosing to Leave the Plan':                              1.282,   # two codes, one bar
    'Drug Plan Quality Improvement':                                   1.628,
    'Monitoring Physical Activity':                                    1.879,
    'Reducing the Risk of Falling':                                    2.287,
}
for name, tab in tableau_bars.items():
    check(f'gap: {name[:40]}', tab,
          sql.loc[sql.MeasureName == name, 'gap_vs_national'].iloc[0],
          pdf.loc[pdf.MeasureName == name, 'gap_vs_national'].iloc[0])

check('bars on the chart', 41, len(sql), len(pdf))
check('measures below the national average', 26,
      (sql.gap_vs_national < 0).sum(), (pdf.gap_vs_national < 0).sum())

print("\nThe two collapsed bars are the check that matters: grouping by Measure Code instead")
print("of Measure Name gives 43 rows and the Call Center bar comes out at -0.164, not +0.213.")

41 bars (Tableau status bar reads "41 marks")

                              MeasureName  la_care_stars  national_avg  gap_vs_national
 Medication Reconciliation Post-Discharge            1.0         3.831           -2.831
                        Care Coordination            1.0         3.495           -2.495
                         Customer Service            1.0         3.467           -2.467
Care for Older Adults – Medication Review            2.0         4.157           -2.157
                      Transitions of Care            1.0         3.114           -2.114
              Plan All-Cause Readmissions            1.0         2.947           -1.947
  Care for Older Adults – Pain Assessment            2.0         3.921           -1.921
              Reviewing Appeals Decisions            2.0         3.713           -1.713

...

                       MeasureName  la_care_stars  national_avg  gap_vs_national
Members Choosing to Leave the Plan            5.0         3.718           

## 5. `Improving Underneath` - the flat overall that hides real gains

The finding the dashboard is built around: between 2025 and 2026 Part C rose 2.5 to 3.0 and Part D rose 3.5 to 4.0, yet the Overall star never moved off 3.0. CMS weights the two halves and rounds to the nearest half star, so half a star of movement on each side can vanish.

- **SQL approach:** an unpivot, one row per part per year, on the single L.A. Care contract. No aggregation is needed because there is exactly one row per contract per year.
- **pandas approach:** `melt` the three star columns into long form, which is what `Measure Values` and `Measure Names` do inside Tableau.

In [6]:
sql = con.execute("""
    SELECT Year, part, stars FROM (
        UNPIVOT (SELECT Year, Overall_Stars, PartC_Stars, PartD_Stars
                 FROM summary WHERE Contract = ? AND Year >= 2025)
        ON Overall_Stars, PartC_Stars, PartD_Stars
        INTO NAME part VALUE stars
    )
    ORDER BY part, Year
""", [LA]).df()

pdf = (summary[(summary.Contract == LA) & (summary.Year >= 2025)]
       .melt(id_vars='Year',
             value_vars=['Overall_Stars', 'PartC_Stars', 'PartD_Stars'],
             var_name='part', value_name='stars')
       .sort_values(['part', 'Year']))

print(sql.to_string(index=False))

def one(df, part, yr):
    return df[(df.part == part) & (df.Year == yr)].stars.iloc[0]

for part, y25, y26 in [('PartC_Stars', 2.5, 3.0), ('PartD_Stars', 3.5, 4.0), ('Overall_Stars', 3.0, 3.0)]:
    check(f'{part} 2025', y25, one(sql, part, 2025), one(pdf, part, 2025))
    check(f'{part} 2026', y26, one(sql, part, 2026), one(pdf, part, 2026))

print('\nBoth halves gained 0.5. The overall gained 0.0. That is the chart.')

 Year          part  stars
 2025 Overall_Stars    3.0
 2026 Overall_Stars    3.0
 2025   PartC_Stars    2.5
 2026   PartC_Stars    3.0
 2025   PartD_Stars    3.5
 2026   PartD_Stars    4.0
[PASS] PartC_Stars 2025: tableau=2.5 sql=2.5 pandas=2.5
[PASS] PartC_Stars 2026: tableau=3.0 sql=3.0 pandas=3.0
[PASS] PartD_Stars 2025: tableau=3.5 sql=3.5 pandas=3.5
[PASS] PartD_Stars 2026: tableau=4.0 sql=4.0 pandas=4.0
[PASS] Overall_Stars 2025: tableau=3.0 sql=3.0 pandas=3.0
[PASS] Overall_Stars 2026: tableau=3.0 sql=3.0 pandas=3.0

Both halves gained 0.5. The overall gained 0.0. That is the chart.


## 6. `Parent Org Quality.` - the group, the parameter, and the boolean bridge

Three Tableau ideas stacked on one sheet:

1. **Group.** `Parent Org (group)` merges the duplicate CMS spellings. Without it, `Molina Healthcare, Inc.` and `Molina Healthcare, Inc.,` are two companies and neither clears the threshold.
2. **Parameter.** `Min Rated Contracts` (1 to 10) sets how many rated plans a company needs to appear.
3. **Boolean bridge.** A measure filter's value box will not accept a parameter, so the comparison is wrapped in a calc: `Meets Min Contracts = [Rated Contracts] >= [Min Rated Contracts]`, filtered to True.

`Rated Contracts = COUNTD(IF NOT ISNULL([Overall Stars]) THEN [Contract] END)` counts only *rated* plans, not all plans.

- **SQL approach:** `CASE WHEN` reproduces the group, a `GROUP BY` gives the average and the distinct count, and a `HAVING` clause plays the role of the boolean filter. `HAVING` is the SQL equivalent of a Tableau measure filter: it runs after aggregation.
- **pandas approach:** `.replace()` for the group, `groupby().agg()` for both numbers, then a boolean mask on the count.

In [7]:
def parent_org_table(min_contracts):
    sql = con.execute("""
        SELECT ParentOrgGroup                     AS parent_org,
               COUNT(DISTINCT Contract)           AS rated_contracts,
               AVG(Overall_Stars)                 AS avg_overall_stars
        FROM summary
        WHERE Year = ? AND Overall_Stars IS NOT NULL
        GROUP BY ParentOrgGroup
        HAVING COUNT(DISTINCT Contract) >= ?      -- the boolean bridge, in SQL
        ORDER BY avg_overall_stars DESC
    """, [YEAR, min_contracts]).df()

    d = summary[(summary.Year == YEAR) & summary.Overall_Stars.notna()]
    pdf = (d.groupby('ParentOrgGroup')
             .agg(rated_contracts=('Contract', 'nunique'),
                  avg_overall_stars=('Overall_Stars', 'mean'))
             .reset_index().rename(columns={'ParentOrgGroup': 'parent_org'}))
    pdf = pdf[pdf.rated_contracts >= min_contracts].sort_values('avg_overall_stars', ascending=False)
    return sql, pdf

sql7, pdf7 = parent_org_table(7)
print('Min Rated Contracts = 7 (what the published dashboard shows)\n')
print(sql7.round(2).to_string(index=False))

tableau_min7 = {
    'Kaiser Foundation Health Plan, Inc.': (8,  4.3),
    'Highmark Health':                     (7,  4.1),
    'Devoted Health, Inc.':                (22, 3.9),
    'UnitedHealth Group, Inc.':            (51, 3.8),
    'CVS Health Corporation':              (39, 3.6),
    'Elevance Health, Inc.':               (37, 3.6),
    'Cambia Health Solutions, Inc.':       (7,  3.5),
    'Humana Inc.':                         (31, 3.4),
    'Centene Corporation':                 (47, 3.3),
    'Health Care Service Corporation':     (24, 3.3),
    'Molina Healthcare, Inc.':             (13, 3.3),
}
for org, (n, avg) in tableau_min7.items():
    check(f'{org} rated contracts', n,
          sql7.loc[sql7.parent_org == org, 'rated_contracts'].iloc[0],
          pdf7.loc[pdf7.parent_org == org, 'rated_contracts'].iloc[0])
    check(f'{org} avg star', avg,
          round(sql7.loc[sql7.parent_org == org, 'avg_overall_stars'].iloc[0], 1),
          round(pdf7.loc[pdf7.parent_org == org, 'avg_overall_stars'].iloc[0], 1), places=1)

check('companies shown at min 7', 11, len(sql7), len(pdf7))

# the headline: the biggest companies are NOT the top scorers
top = sql7.iloc[0]
biggest = sql7.sort_values('rated_contracts', ascending=False).iloc[0]
print(f"\ntop scorer : {top.parent_org} ({int(top.rated_contracts)} plans, {top.avg_overall_stars:.2f})")
print(f"biggest    : {biggest.parent_org} ({int(biggest.rated_contracts)} plans, {biggest.avg_overall_stars:.2f})")
assert top.parent_org != biggest.parent_org, 'the headline claims these differ'

Min Rated Contracts = 7 (what the published dashboard shows)

                         parent_org  rated_contracts  avg_overall_stars
Kaiser Foundation Health Plan, Inc.                8               4.31
                    Highmark Health                7               4.14
               Devoted Health, Inc.               22               3.89
           UnitedHealth Group, Inc.               51               3.79
             CVS Health Corporation               39               3.60
              Elevance Health, Inc.               37               3.59
      Cambia Health Solutions, Inc.                7               3.50
                        Humana Inc.               31               3.40
                Centene Corporation               47               3.31
    Health Care Service Corporation               24               3.27
            Molina Healthcare, Inc.               13               3.27
[PASS] Kaiser Foundation Health Plan, Inc. rated contracts: tableau=8 sql=


top scorer : Kaiser Foundation Health Plan, Inc. (8 plans, 4.31)
biggest    : UnitedHealth Group, Inc. (51 plans, 3.79)


## 7. `Who Is Improving` - Difference From, Relative to First

The highlight table shows each company's change in average star **since 2024**, its first year in the data. In Tableau that is a table calculation: `AVG(Overall Stars)` with **Difference From**, **Relative to: First**, **Compute Using: Table (across)**.

The `Big Company` filter is itself a FIXED LOD, not a view-level count:

```
Big Company = { FIXED [Parent Org (group)] :
                COUNTD(IF [Year] = 2026 AND NOT ISNULL([Overall Stars]) THEN [Contract] END) }
              >= [Min Rated Contracts]
```

It has to be FIXED. A view-level `COUNTD` would recount contracts **per year**, so a company that had 6 rated plans in 2024 and 8 in 2026 would appear in some columns and vanish in others, punching holes in its own trend line. FIXED pins the count to the company, once.

- **SQL approach:** `FIRST_VALUE(avg_star) OVER (PARTITION BY parent_org ORDER BY Year)` gives each company its 2024 baseline on every row; subtract to get the difference. `FIRST_VALUE` *is* "Relative to First," and the `PARTITION BY` *is* "Compute Using Table (across)."
- **pandas approach:** group by company and subtract the first year's value with `transform(lambda v: v - v.iloc[0])`.

In [8]:
MIN_CONTRACTS = 7

sql = con.execute("""
    WITH big AS (                                   -- the FIXED [Parent Org] LOD
        SELECT ParentOrgGroup
        FROM summary
        WHERE Year = 2026 AND Overall_Stars IS NOT NULL
        GROUP BY ParentOrgGroup
        HAVING COUNT(DISTINCT Contract) >= ?
    ),
    yearly AS (
        SELECT s.ParentOrgGroup AS parent_org,
               s.Year,
               AVG(s.Overall_Stars) AS avg_star
        FROM summary s
        JOIN big USING (ParentOrgGroup)
        WHERE s.Overall_Stars IS NOT NULL
        GROUP BY 1, 2
    )
    SELECT parent_org, Year, avg_star,
           avg_star - FIRST_VALUE(avg_star) OVER (       -- Difference From, Relative to First
               PARTITION BY parent_org ORDER BY Year
           ) AS diff_since_first
    FROM yearly
    ORDER BY parent_org, Year
""", [MIN_CONTRACTS]).df()

d = summary[summary.Overall_Stars.notna()]
big = (d[d.Year == 2026].groupby('ParentOrgGroup').Contract.nunique()
         .loc[lambda x: x >= MIN_CONTRACTS].index)
pdf = (d[d.ParentOrgGroup.isin(big)]
       .groupby(['ParentOrgGroup', 'Year'], as_index=False)
       .Overall_Stars.mean()
       .rename(columns={'ParentOrgGroup': 'parent_org', 'Overall_Stars': 'avg_star'})
       .sort_values(['parent_org', 'Year']))
pdf['diff_since_first'] = (pdf.groupby('parent_org').avg_star
                              .transform(lambda v: v - v.iloc[0]))

wide = sql.pivot(index='parent_org', columns='Year', values='diff_since_first').round(1)
print(wide.sort_values(2026, ascending=False).to_string())

tableau_2026 = {
    'Kaiser Foundation Health Plan, Inc.':  0.5,
    'Centene Corporation':                  0.4,
    'Molina Healthcare, Inc.':              0.3,
    'Health Care Service Corporation':      0.0,
    'Elevance Health, Inc.':                0.0,
    'Cambia Health Solutions, Inc.':        0.0,
    'CVS Health Corporation':              -0.2,
    'Highmark Health':                     -0.2,
    'UnitedHealth Group, Inc.':            -0.1,
    'Humana Inc.':                         -0.5,
    'Devoted Health, Inc.':                -0.5,
}
def cell(df, org, yr):
    return round(df[(df.parent_org == org) & (df.Year == yr)].diff_since_first.iloc[0], 1)

for org, tab in tableau_2026.items():
    check(f'{org} change since 2024', tab, cell(sql, org, 2026), cell(pdf, org, 2026), places=1)

# every company's 2024 column must be exactly zero: that is what "Relative to First" means
assert (sql[sql.Year == 2024].diff_since_first.abs() < 1e-9).all()
print('\n[PASS] every 2024 cell is 0.0, which is what Relative to First guarantees')

# the headline
risers  = [o for o, v in tableau_2026.items() if v > 0]
fallers = [o for o, v in tableau_2026.items() if v < 0]
lowest  = sql7.sort_values('avg_overall_stars').head(3).parent_org.tolist()
print(f'\nrising : {risers}')
print(f'falling: {fallers}')
print(f'lowest scorers: {lowest}')
assert 'Centene Corporation' in risers and 'Molina Healthcare, Inc.' in risers
assert 'Centene Corporation' in lowest and 'Molina Healthcare, Inc.' in lowest
print("[PASS] headline holds: L.A. Care's nearest rivals (Centene, Molina) are among the lowest scorers AND are climbing")

Year                                 2024  2025  2026
parent_org                                           
Kaiser Foundation Health Plan, Inc.   0.0   0.5   0.5
Centene Corporation                   0.0   0.3   0.4
Molina Healthcare, Inc.               0.0   0.1   0.3
Elevance Health, Inc.                 0.0  -0.1   0.0
Cambia Health Solutions, Inc.         0.0  -0.1   0.0
Health Care Service Corporation       0.0   0.2   0.0
UnitedHealth Group, Inc.              0.0  -0.2  -0.1
CVS Health Corporation                0.0   0.1  -0.2
Highmark Health                       0.0   0.1  -0.2
Devoted Health, Inc.                  0.0  -0.4  -0.5
Humana Inc.                           0.0  -0.3  -0.5
[PASS] Kaiser Foundation Health Plan, Inc. change since 2024: tableau=0.5 sql=0.5 pandas=0.5
[PASS] Centene Corporation change since 2024: tableau=0.4 sql=0.4 pandas=0.4
[PASS] Molina Healthcare, Inc. change since 2024: tableau=0.3 sql=0.3 pandas=0.3
[PASS] Health Care Service Corporation change s

## 8. Context filters - the one Tableau idea with no obvious pandas analog

A **context filter** changes *when* a filter runs. Ordinary dimension filters run **after** FIXED LODs. A context filter runs **before** them.

The SQL translation is exact and worth memorising:

> **Add to Context = move the condition into the CTE.**

Below, the same benchmark is computed twice. The only difference is which side of the CTE boundary the `WHERE Contract = 'H1224'` sits on.

- **Without context** (`WHERE` outside the CTE): the window sees all 516 contracts, so the benchmark is 3.65. This is what the dashboard shows.
- **With context** (`WHERE` inside the CTE): the window only ever sees L.A. Care, so the benchmark collapses to L.A. Care's own 3.0 and the two lines lie on top of each other. The chart becomes meaningless.

This is the pitfall to demonstrate in the blog: adding a filter to context is not a performance tweak, it silently changes what a FIXED LOD can see.

In [9]:
without_context = con.execute("""
    WITH benchmark AS (
        SELECT Year, Contract, AVG(Overall_Stars) OVER (PARTITION BY Year) AS national
        FROM summary                                     -- no filter here
    )
    SELECT DISTINCT national FROM benchmark
    WHERE Year = ? AND Contract = ?                      -- filter AFTER the window
""", [YEAR, LA]).df().national.iloc[0]

with_context = con.execute("""
    WITH benchmark AS (
        SELECT Year, Contract, AVG(Overall_Stars) OVER (PARTITION BY Year) AS national
        FROM summary
        WHERE Contract = ?                               -- filter INSIDE the CTE = Add to Context
    )
    SELECT DISTINCT national FROM benchmark
    WHERE Year = ?
""", [LA, YEAR]).df().national.iloc[0]

# pandas: the same distinction, expressed as when you filter the frame
pd_without = summary.groupby('Year').Overall_Stars.mean().loc[YEAR]
pd_with    = summary[summary.Contract == LA].groupby('Year').Overall_Stars.mean().loc[YEAR]

print(f'benchmark WITHOUT context : sql {without_context:.4f}   pandas {pd_without:.4f}   <- 516 contracts')
print(f'benchmark WITH context    : sql {with_context:.4f}   pandas {pd_with:.4f}   <- 1 contract, the LOD is blinded')

check('benchmark without context', 3.6521, without_context, pd_without, places=4)
check('benchmark with context (the pitfall)', 3.0, with_context, pd_with)

print('\nSame calc. Same data. The filter moved one line up, and the answer changed by 0.65 stars.')

benchmark WITHOUT context : sql 3.6521   pandas 3.6521   <- 516 contracts
benchmark WITH context    : sql 3.0000   pandas 3.0000   <- 1 contract, the LOD is blinded
[PASS] benchmark without context: tableau=3.6521 sql=3.6521 pandas=3.6521
[PASS] benchmark with context (the pitfall): tableau=3.0 sql=3.0 pandas=3.0

Same calc. Same data. The filter moved one line up, and the answer changed by 0.65 stars.


## 9. `Gap Explorer` - a reference join, and a parameter that swaps the measure

The Star Ratings tables do not say how much each measure counts, but CMS does not average them, it weights them from 1 to 5. Those weights live in a separate PDF and are keyed into `measure_weights.csv` in `nb01`.

Three Tableau ideas stack on this one sheet:

1. **A relationship** joining `measure_weights.csv` onto the measure data, on `Measure Code` **and** `Year`. Both keys matter: weights change between rating years, and the reference file holds only 2026.
2. **A weighted gap**: `Weighted Gap = [Gap vs National] * AVG([Weight])`
3. **A parameter that swaps which measure the bars plot**:

```
Rank By          -> a String parameter, list: "Raw gap" | "Weighted gap"
Gap (Rank By)    -> IF [Rank By] = "Raw gap" THEN [Gap vs National] ELSE [Weighted Gap] END
```

The third one is the pattern worth keeping. A parameter cannot itself be plotted; it has to be read by a calculated field, and that field is what goes on the shelf. The chart is then plotting one field whose *definition* changes.

- **SQL approach:** the join is an ordinary `JOIN ... USING (MeasureCode, Year)`. The parameter swap is a `CASE` in the `SELECT` list, driven by a bind variable.
- **pandas approach:** `merge` on both keys, then pick the column by name from a variable.

In [10]:
weights = pd.read_csv(DATA / 'measure_weights.csv')
con.register('weights', weights)

def gap_explorer(rank_by):
    """rank_by: 'Raw gap' or 'Weighted gap' - the Tableau parameter, as a bind variable."""
    sql = con.execute("""
        WITH benchmark AS (                              -- the FIXED LOD, per Year and Measure Code
            SELECT Year, Contract, MeasureCode, MeasureName, Stars,
                   AVG(Stars) OVER (PARTITION BY Year, MeasureCode) AS national
            FROM measures
            WHERE Stars IS NOT NULL
        ),
        joined AS (                                      -- the relationship: BOTH keys
            SELECT b.*, w.Weight, w.WeightCategory, w.DataSource
            FROM benchmark b
            JOIN weights w USING (MeasureCode, Year)
        ),
        view AS (                                        -- the view: aggregated to Measure NAME
            SELECT MeasureName,
                   AVG(Stars)                                AS la_care,
                   AVG(national)                             AS national,
                   AVG(Weight)                               AS weight,
                   AVG(Stars) - AVG(national)                AS raw_gap
            FROM joined
            WHERE Contract = ? AND Year = ?
            GROUP BY MeasureName
        )
        SELECT MeasureName, la_care, national, weight, raw_gap,
               raw_gap * weight AS weighted_gap,
               CASE WHEN ? = 'Raw gap'                       -- the parameter swap
                    THEN raw_gap
                    ELSE raw_gap * weight
               END AS gap_rank_by
        FROM view
        ORDER BY gap_rank_by
    """, [LA, YEAR, rank_by]).df()

    m = measures[measures.Stars.notna()].copy()
    m['national'] = m.groupby(['Year', 'MeasureCode'])['Stars'].transform('mean')
    m = m.merge(weights[['Year', 'MeasureCode', 'Weight']], on=['Year', 'MeasureCode'])   # both keys
    v = (m[(m.Contract == LA) & (m.Year == YEAR)]
         .groupby('MeasureName')
         .agg(la_care=('Stars', 'mean'), national=('national', 'mean'), weight=('Weight', 'mean'))
         .reset_index())
    v['raw_gap']      = v.la_care - v.national
    v['weighted_gap'] = v.raw_gap * v.weight
    v['gap_rank_by']  = v.raw_gap if rank_by == 'Raw gap' else v.weighted_gap    # the parameter swap
    v = v.sort_values('gap_rank_by')
    return sql, v

raw_sql, raw_pd = gap_explorer('Raw gap')
wtd_sql, wtd_pd = gap_explorer('Weighted gap')

print('Rank By = "Raw gap"        top bar:', raw_sql.MeasureName.iloc[0], round(raw_sql.gap_rank_by.iloc[0], 3))
print('Rank By = "Weighted gap"   top bar:', wtd_sql.MeasureName.iloc[0], round(wtd_sql.gap_rank_by.iloc[0], 3))
print()
print(wtd_sql.head(6)[['MeasureName', 'la_care', 'national', 'raw_gap', 'weight', 'weighted_gap']].round(3).to_string(index=False))

# the join must not change the number of bars
check('bars, Rank By = Raw gap',      41, len(raw_sql), len(raw_pd))
check('bars, Rank By = Weighted gap', 41, len(wtd_sql), len(wtd_pd))

# the exact bar labels Tableau renders on Gap Explorer, in each toggle position
tableau_weighted = {
    'Plan All-Cause Readmissions':                    -5.84,
    'Care Coordination':                              -4.99,
    'Customer Service':                               -4.93,
    'Medication Adherence for Cholesterol (Statins)': -3.63,
    'Reviewing Appeals Decisions':                    -3.43,
    'Medication Reconciliation Post-Discharge':       -2.83,
    'Drug Plan Quality Improvement':                   8.14,
}
for name, tab in tableau_weighted.items():
    check(f'weighted gap: {name[:38]}', tab,
          wtd_sql.loc[wtd_sql.MeasureName == name, 'weighted_gap'].iloc[0],
          wtd_pd.loc[wtd_pd.MeasureName == name, 'weighted_gap'].iloc[0], places=2)

# the toggle must actually REORDER, not merely rescale. that is the whole point of the chart.
assert raw_sql.MeasureName.iloc[0] == 'Medication Reconciliation Post-Discharge'
assert wtd_sql.MeasureName.iloc[0] == 'Plan All-Cause Readmissions'
print('\n[PASS] the toggle reorders: the biggest raw gap is NOT the biggest weighted gap')

print('\n=== how the top five reorder ===')
for i, (r, w) in enumerate(zip(raw_sql.MeasureName.head(5), wtd_sql.MeasureName.head(5)), 1):
    print(f'{i}. raw: {r[:42]:44s} weighted: {w[:42]:44s}{"" if r == w else "  <- changed"}')

Rank By = "Raw gap"        top bar: Medication Reconciliation Post-Discharge -2.831
Rank By = "Weighted gap"   top bar: Plan All-Cause Readmissions -5.842

                                   MeasureName  la_care  national  raw_gap  weight  weighted_gap
                   Plan All-Cause Readmissions      1.0     2.947   -1.947     3.0        -5.842
                             Care Coordination      1.0     3.495   -2.495     2.0        -4.990
                              Customer Service      1.0     3.467   -2.467     2.0        -4.935
Medication Adherence for Cholesterol (Statins)      2.0     3.210   -1.210     3.0        -3.630
                   Reviewing Appeals Decisions      2.0     3.713   -1.713     2.0        -3.426
         Getting Appointments and Care Quickly      2.0     3.550   -1.550     2.0        -3.100
[PASS] bars, Rank By = Raw gap: tableau=41 sql=41.0 pandas=41.0
[PASS] bars, Rank By = Weighted gap: tableau=41 sql=41.0 pandas=41.0
[PASS] weighted gap: Plan All-Ca

## Parity summary

In [11]:
out = pd.DataFrame(RESULTS, columns=['figure', 'tableau', 'duckdb_sql', 'pandas', 'match'])
out['duckdb_sql'] = out.duckdb_sql.astype(float).round(4)
out['pandas']     = out['pandas'].astype(float).round(4)

print(out.to_string(index=False))
print()
print(f'{out.match.sum()} of {len(out)} checks pass')
assert out.match.all(), 'a figure on the dashboard does not reconcile'
print('ALL PARITY CHECKS PASS - every published figure reconciles in Tableau, DuckDB SQL, and pandas')

                                               figure  tableau  duckdb_sql   pandas  match
                               National Avg Star 2024   3.6817      3.6817   3.6817   True
                               National Avg Star 2025   3.6497      3.6497   3.6497   True
                               National Avg Star 2026   3.6521      3.6521   3.6521   True
                          L.A. Care overall star 2025   3.0000      3.0000   3.0000   True
                          L.A. Care overall star 2026   3.0000      3.0000   3.0000   True
                               contracts at 2.0 stars   2.0000      2.0000   2.0000   True
                               contracts at 2.5 stars  21.0000     21.0000  21.0000   True
                               contracts at 3.0 stars 111.0000    111.0000 111.0000   True
                               contracts at 3.5 stars 175.0000    175.0000 175.0000   True
                               contracts at 4.0 stars 116.0000    116.0000 116.0000   True